# Phase 6 — Risk stress (P90 / downside yield)

**Phase 6 risk:** Full interannual distribution — P90 from **5th percentile** of year-level stress ratios (not worst-2-year mean).

| Input | Output |
|-------|--------|
| `scratch/weather_cell_year/*.parquet` | `cell_p90_yield.parquet` |
| `cell_p50_yield.parquet` | `cell_risk_spread.parquet` |

**Stress ratio (per year):** `year_resource / climatology` → **P90 ratio = 5th percentile** across 8 years → `P90 = P50 × ratio` (so P90 ≤ P50).

**Extra:** `solar_stress_ratio_std`, `wind_stress_ratio_std` for composite S (notebook 13).

**Requires:** `01`, `06`, `10`

In [1]:
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import geopandas as gpd
import plotly.graph_objects as go
import polars as pl

ROOT = Path('../..').resolve()
SCRATCH_DIR = ROOT / 'data' / 'processed' / 'scratch' / 'weather_cell_year'
GATEWAY_PATH = ROOT / 'data' / 'processed' / 'cells_passing_gateway.parquet'
P50_PATH = ROOT / 'data' / 'processed' / 'cell_p50_yield.parquet'
HYBRID_PATH = ROOT / 'data' / 'processed' / 'hybrid_suitability.parquet'
P90_PATH = ROOT / 'data' / 'processed' / 'cell_p90_yield.parquet'
RISK_PATH = ROOT / 'data' / 'processed' / 'cell_risk_spread.parquet'
REPORT_PATH = ROOT / 'data' / 'processed' / 'risk_stress_report.json'
MAP_PATH = ROOT / 'maps' / 'risk_spread_ny.html'
NY_BOUNDARY_PATH = ROOT / 'data' / 'ref' / 'ny_state_boundary.geojson'

STRESS_PERCENTILE = 0.05  # 5th percentile of year-level ratios = downside P90
print('ROOT:', ROOT)

ROOT: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York


In [2]:
def year_level_stress(yearly: pl.DataFrame, clim: pl.DataFrame, value_col: str, prefix: str) -> pl.DataFrame:
    """Per-cell 5th-percentile and std of annual resource / climatology ratios."""
    ratios = (
        yearly.select(['latitude', 'longitude', 'year', value_col])
        .join(
            clim.select(['latitude', 'longitude', pl.col(value_col).alias('clim_value')]),
            on=['latitude', 'longitude'],
        )
        .with_columns(
            (pl.col(value_col) / pl.col('clim_value')).clip(0, 1).alias('stress_ratio')
        )
    )
    return ratios.group_by(['latitude', 'longitude']).agg(
        pl.col('stress_ratio').quantile(STRESS_PERCENTILE).alias(f'{prefix}_stress_ratio'),
        pl.col('stress_ratio').std().alias(f'{prefix}_stress_ratio_std'),
        pl.col('year').n_unique().alias(f'{prefix}_stress_n_years'),
    )


year_files = sorted(SCRATCH_DIR.glob('weather_cell_*.parquet'))
if not year_files:
    raise FileNotFoundError(f'Run 01 first: {SCRATCH_DIR}')

yearly = pl.concat([
    pl.read_parquet(p).with_columns(pl.lit(int(p.stem.split('_')[-1])).alias('year'))
    for p in year_files
])
print(f'{len(year_files)} years, {yearly.height:,} cell-years')

cells = pl.read_parquet(GATEWAY_PATH).filter(pl.col('in_ny_state'))
clim = cells.select([
    'latitude', 'longitude', 'shortwave_mean', 'wind_power_mean',
])

s_stress = year_level_stress(yearly, clim, 'shortwave_mean', 'solar')
w_stress = year_level_stress(yearly, clim, 'wind_power_mean', 'wind')
print('Computed full-year stress ratios (p5 + std)')

8 years, 26,999 cell-years
Computed full-year stress ratios (p5 + std)


In [3]:
out = (
    clim
    .join(s_stress, on=['latitude', 'longitude'], how='left')
    .join(w_stress, on=['latitude', 'longitude'], how='left')
)

if P50_PATH.exists():
    out = out.join(pl.read_parquet(P50_PATH), on=['latitude', 'longitude'], how='left')
else:
    raise FileNotFoundError(f'Run 10 first: {P50_PATH}')

out = out.with_columns([
    (pl.col('solar_cf_p50') * pl.col('solar_stress_ratio')).alias('solar_cf_p90'),
    (pl.col('wind_cf_p50') * pl.col('wind_stress_ratio')).alias('wind_cf_p90'),
]).with_columns([
    (pl.col('solar_cf_p50') - pl.col('solar_cf_p90')).alias('solar_risk_spread'),
    (pl.col('wind_cf_p50') - pl.col('wind_cf_p90')).alias('wind_risk_spread'),
]).with_columns([
    ((pl.col('solar_risk_spread') + pl.col('wind_risk_spread')) / 2).alias('hybrid_risk_spread'),
])

if HYBRID_PATH.exists():
    out = out.join(
        pl.read_parquet(HYBRID_PATH).select('latitude', 'longitude', 'hybrid_seasonal_score', 'jackpot_zone'),
        on=['latitude', 'longitude'], how='left',
    )

out.write_parquet(P90_PATH)
out.select([
    'latitude', 'longitude',
    'solar_stress_ratio', 'solar_stress_ratio_std',
    'wind_stress_ratio', 'wind_stress_ratio_std',
    'solar_cf_p50', 'solar_cf_p90', 'solar_risk_spread',
    'wind_cf_p50', 'wind_cf_p90', 'wind_risk_spread', 'hybrid_risk_spread',
]).write_parquet(RISK_PATH)
print(f'Wrote {P90_PATH} ({out.height} rows)')

Wrote /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/cell_p90_yield.parquet (1089 rows)


In [4]:
valid = out.filter(pl.col('solar_cf_p50').is_not_null() & pl.col('solar_cf_p90').is_not_null())
pct_ok = float((valid['solar_cf_p90'] <= valid['solar_cf_p50']).sum() / max(valid.height, 1) * 100)

report = {
    'n_cells': out.height,
    'method': 'full_year_p5',
    'stress_percentile': STRESS_PERCENTILE,
    'v1_baseline_mean_hybrid_risk_spread': 0.001,
    'pct_solar_p90_le_p50': pct_ok,
    'mean_hybrid_risk_spread': float(out['hybrid_risk_spread'].mean()),
    'mean_solar_stress_ratio_std': float(out['solar_stress_ratio_std'].mean()),
    'mean_wind_stress_ratio_std': float(out['wind_stress_ratio_std'].mean()),
    'top_risk': out.sort('hybrid_risk_spread', descending=True).head(5).to_dicts(),
}
REPORT_PATH.write_text(json.dumps(report, indent=2, default=float), encoding='utf-8')
print(f'Solar P90 ≤ P50: {pct_ok:.1f}% (target ≥95%)')
print(f'Mean hybrid risk spread: {report["mean_hybrid_risk_spread"]:.4f} (R5 target > 0.005)')
out.sort('hybrid_risk_spread', descending=True).head(5)

Solar P90 ≤ P50: 100.0% (target ≥95%)
Mean hybrid risk spread: 0.0059 (R5 target > 0.005)


latitude,longitude,shortwave_mean,wind_power_mean,solar_stress_ratio,solar_stress_ratio_std,solar_stress_n_years,wind_stress_ratio,wind_stress_ratio_std,wind_stress_n_years,solar_cf_p50_v1,solar_cf_p50_v2,solar_cf_p50,wind_cf_p50_v1,wind_cf_p50_v2,wind_cf_p50,hybrid_cf_p50,solar_cf_p90,wind_cf_p90,solar_risk_spread,wind_risk_spread,hybrid_risk_spread,hybrid_seasonal_score,jackpot_zone
f64,f64,f64,f64,f64,f64,u32,f64,f64,u32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,bool
43.56,-76.2,253.240899,165.268527,0.178261,0.335474,6,0.49624,0.223321,6,0.040637,0.034743,0.039164,0.027867,0.047659,0.032815,0.035989,0.006981,0.016284,0.032182,0.016531,0.024357,0.907502,true
42.39,-77.55,296.991333,67.134654,0.935545,0.023087,8,0.761029,0.077135,8,0.045927,0.038998,0.044194,0.13775,0.113855,0.131776,0.087985,0.041346,0.100285,0.002849,0.031491,0.01717,0.445445,false
42.03,-77.64,302.029109,90.962273,0.933015,0.023265,8,0.796073,0.069445,8,0.030583,0.04987,0.035405,0.135105,0.091943,0.124314,0.079859,0.033033,0.098963,0.002372,0.025351,0.013861,0.754201,true
41.58,-74.22,307.590826,36.881548,0.923534,0.026362,8,0.732395,0.093089,8,0.031455,0.02396,0.029582,0.108094,0.048933,0.093304,0.061443,0.02732,0.068335,0.002262,0.024969,0.013615,0.032031,false
41.67,-74.13,306.914329,35.120471,0.924033,0.026147,8,0.744148,0.088361,8,0.038483,0.03376,0.037302,0.108094,0.048933,0.093304,0.065303,0.034468,0.069432,0.002834,0.023872,0.013353,0.028965,false


In [5]:
m = out.to_pandas()
ny = gpd.read_file(NY_BOUNDARY_PATH)
outline = ny.boundary.iloc[0]

fig = go.Figure()
fig.add_trace(go.Scattermap(
    lat=m['latitude'], lon=m['longitude'], mode='markers',
    marker=dict(size=8, color=m['hybrid_risk_spread'], colorscale='Reds', showscale=True, opacity=0.7),
))
fig.add_trace(go.Scattermap(
    lat=list(outline.xy[1]), lon=list(outline.xy[0]),
    mode='lines', line=dict(width=2, color='#333'), hoverinfo='skip', showlegend=False,
))
fig.update_layout(
    map=dict(style='carto-positron', center=dict(lat=42.9, lon=-75.8), zoom=5.5),
    title='Hybrid risk spread (P50 − P90 stress weather)',
    height=800,
)
MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(str(MAP_PATH), include_plotlyjs='cdn')
print(f'Map -> {MAP_PATH}')

Map -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/maps/risk_spread_ny.html
